# 02 - Data Cleaning

## Purpose

This notebook applies cleaning rules to the raw Online Retail dataset and creates an analysis-ready sales dataset.

The cleaning process includes:
- removing exact duplicates
- creating revenue fields
- identifying cancellations and returns
- handling missing customer and product information
- excluding invalid records from the clean sales dataset
- exporting cleaned data for SQL analysis and dashboarding

In [1]:
import pandas as pd
from pathlib import Path

raw_file = Path("../data/raw/Online Retail.xlsx")

df_raw = pd.read_excel(raw_file)

df_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
df_raw.shape

(541909, 8)

In [3]:
df = df_raw.copy()

In [4]:
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns.tolist()

['invoiceno',
 'stockcode',
 'description',
 'quantity',
 'invoicedate',
 'unitprice',
 'customerid',
 'country']

In [5]:
rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)

duplicates_removed = rows_before - rows_after

duplicates_removed

5268

In [6]:
df["revenue"] = df["quantity"] * df["unitprice"]

df[["quantity", "unitprice", "revenue"]].describe().round()

,quantity,unitprice,revenue
count,536641.0,536641.0,536641.0
mean,10.0,5.0,18.0
std,219.0,97.0,381.0
min,-80995.0,-11062.0,-168470.0
25%,1.0,1.0,4.0
50%,3.0,2.0,10.0
75%,10.0,4.0,17.0
max,80995.0,38970.0,168470.0


In [7]:
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns.tolist()

['invoiceno',
 'stockcode',
 'description',
 'quantity',
 'invoicedate',
 'unitprice',
 'customerid',
 'country',
 'revenue']

In [8]:
df["transaction_type"] = df["invoiceno"].astype(str).str.startswith("C").map({
    True: "Cancellation",
    False: "Sale"
})

df["transaction_type"].value_counts()

transaction_type
Sale            527390
Cancellation      9251
Name: count, dtype: int64

In [9]:
df[df["unitprice"] < 0]

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06,Sale
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06,Sale


In [10]:
df[df["unitprice"] == 0].head(20)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,0.0,Sale
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,0.0,Sale
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0,Sale
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0,Sale
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0,Sale
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0,Sale
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0,Sale
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom,0.0,Sale
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom,0.0,Sale
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom,-0.0,Sale


In [11]:
df[(df["quantity"] < 0) & (df["transaction_type"] == "Sale")].head(20)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom,-0.0,Sale
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom,-0.0,Sale
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom,-0.0,Sale
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom,-0.0,Sale
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom,-0.0,Sale
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom,-0.0,Sale
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom,-0.0,Sale
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom,-0.0,Sale
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom,-0.0,Sale
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom,-0.0,Sale


In [12]:
df_raw.shape

(541909, 8)

In [13]:
duplicates_removed

5268

In [14]:
df["transaction_type"].value_counts()

transaction_type
Sale            527390
Cancellation      9251
Name: count, dtype: int64

In [15]:
df[df["unitprice"] < 0]

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06,Sale
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06,Sale


In [16]:
df[(df["quantity"] < 0) & (df["transaction_type"] == "Sale")].shape

(1336, 10)

In [17]:
negative_sales = df[(df["quantity"] < 0) & (df["transaction_type"] == "Sale")]

negative_sales.shape

(1336, 10)

In [18]:
negative_sales[["quantity", "unitprice", "revenue"]].describe()

,quantity,unitprice,revenue
count,1336.000000,1336.0,1336.0
mean,-154.907934,0.0,0.0
std,588.292456,0.0,0.0
min,-9600.000000,0.0,-0.0
25%,-84.000000,0.0,-0.0
50%,-30.000000,0.0,-0.0
75%,-8.000000,0.0,0.0
max,-1.000000,0.0,-0.0


In [19]:
df.columns.tolist()

['invoiceno',
 'stockcode',
 'description',
 'quantity',
 'invoicedate',
 'unitprice',
 'customerid',
 'country',
 'revenue',
 'transaction_type']

In [20]:
df["transaction_type"].value_counts()

transaction_type
Sale            527390
Cancellation      9251
Name: count, dtype: int64

In [21]:
negative_sales.shape

(1336, 10)

In [22]:
negative_sales["description"].value_counts().head(20)

description
check                         120
damages                        45
damaged                        42
?                              41
sold as set on dotcom          20
Damaged                        14
thrown away                     9
Unsaleable, destroyed.          9
??                              7
damages?                        5
wet damaged                     5
ebay                            5
smashed                         4
missing                         3
CHECK                           3
wet pallet                      3
Dotcom sales                    2
reverse 21/5/10 adjustment      2
counted                         2
crushed                         2
Name: count, dtype: int64

In [23]:
df[(df["quantity"] > 0) & (df["unitprice"] > 0) & (df["transaction_type"] == "Sale")].shape

(524878, 10)

In [24]:
df[(df["transaction_type"] == "Cancellation") | (df["quantity"] < 0) & (df["revenue"] < 0)].shape

(9251, 10)

## Accounting Adjustment Exclusion

During SQL KPI validation, accounting adjustment records were found inside the clean sales dataset.

These records had descriptions such as "Adjust bad debt" and invoice numbers starting with "A". Since they do not represent normal customer product sales, they were excluded from the clean sales dataset and moved into excluded records.

This prevents executive sales KPIs from being inflated by non-sales accounting adjustments.

In [25]:
accounting_adjustments = df[
    df["description"].str.contains("adjust bad debt", case=False, na=False) |
    (df["stockcode"].astype(str).str.upper() == "B") |
    (df["invoiceno"].astype(str).str.startswith("A"))
].copy()

accounting_adjustments

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,11062.06,Sale
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06,Sale
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06,Sale


In [26]:
accounting_adjustments.shape

(3, 10)

In [27]:
clean_sales = df[
    (df["transaction_type"] == "Sale") &
    (df["quantity"] > 0) &
    (df["unitprice"] > 0) &
    (df["revenue"] > 0) &
    ~(
        df["description"].str.contains("adjust bad debt", case=False, na=False) |
        (df["stockcode"].astype(str).str.upper() == "B") |
        (df["invoiceno"].astype(str).str.startswith("A"))
    )
].copy()

returns_cancellations = df[
    (df["transaction_type"] == "Cancellation") |
    (df["quantity"] < 0) |
    (df["revenue"] < 0)
].copy()

excluded_records = df[
    ~df.index.isin(clean_sales.index) &
    ~df.index.isin(returns_cancellations.index)
].copy()

In [28]:
print("Clean sales:", clean_sales.shape)
print("Returns/cancellations:", returns_cancellations.shape)
print("Excluded records:", excluded_records.shape)
print("Total accounted rows:", len(clean_sales) + len(returns_cancellations) + len(excluded_records))
print("Original deduplicated rows:", len(df))

Clean sales: (524877, 10)
Returns/cancellations: (10589, 10)
Excluded records: (1175, 10)
Total accounted rows: 536641
Original deduplicated rows: 536641


In [29]:
for dataset in [clean_sales, returns_cancellations, excluded_records]:
    dataset["customerid"] = dataset["customerid"].apply(
        lambda x: None if pd.isna(x) else str(int(float(x)))
    )
    dataset["invoiceno"] = dataset["invoiceno"].astype(str)
    dataset["stockcode"] = dataset["stockcode"].astype(str)

In [30]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

clean_sales.to_csv(processed_path / "clean_sales.csv", index=False)
returns_cancellations.to_csv(processed_path / "returns_cancellations.csv", index=False)
excluded_records.to_csv(processed_path / "excluded_records.csv", index=False)

sample_clean_sales = clean_sales.sample(n=10000, random_state=42)
sample_clean_sales.to_csv(processed_path / "sample_clean_sales.csv", index=False)

In [31]:
pd.read_csv(
    processed_path / "clean_sales.csv",
    dtype={"customerid": str, "invoiceno": str, "stockcode": str}
).head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,Sale
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,Sale
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale


In [36]:
pd.read_csv(
    processed_path / "returns_cancellations.csv",
    dtype={"customerid": str, "invoiceno": str, "stockcode": str}
).head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
0,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527,United Kingdom,-27.50,Cancellation
1,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311,United Kingdom,-4.65,Cancellation
2,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548,United Kingdom,-19.80,Cancellation
3,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,-6.96,Cancellation
4,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,-6.96,Cancellation


In [37]:
pd.read_csv(
    processed_path / "excluded_records.csv",
    dtype={"customerid": str, "invoiceno": str, "stockcode": str}
).head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
0,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,0.0,Sale
1,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,0.0,Sale
2,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0,Sale
3,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0,Sale
4,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0,Sale


In [38]:
print("Deduplicated dataset rows:", len(df))
print("Clean sales rows:", len(clean_sales))
print("Returns/cancellations rows:", len(returns_cancellations))
print("Excluded records rows:", len(excluded_records))
print("Total accounted rows:", len(clean_sales) + len(returns_cancellations) + len(excluded_records))
print("Difference:", len(df) - (len(clean_sales) + len(returns_cancellations) + len(excluded_records)))

Deduplicated dataset rows: 536641
Clean sales rows: 524877
Returns/cancellations rows: 10589
Excluded records rows: 1175
Total accounted rows: 536641
Difference: 0


In [39]:
accounting_adjustments.shape

(3, 10)